In [117]:
import pandas as pd

save_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/refact-effort/monomorph_metrics2.csv"
df = pd.read_csv(save_path)

In [118]:
import pandas as pd
import numpy as np

def format_rate_col(row, generated=True):
    if generated:
        updated_col = 'new_relevant_artifacts_updated'
        all_col = 'all_new_relevant_artifacts'
        rate_col = 'GAR-R'
    else:
        updated_col = 'old_relevant_artifacts_updated'
        all_col = 'all_old_relevant_artifacts'
        rate_col = 'MAR-R'
    return f"{int(row[updated_col])} of {int(row[all_col])} ({row[rate_col]*100:.1f}%)" if row[all_col] > 0 else "0 of 0 (0.0%)"

def extract_metrics(df):
    """
    Extract metrics from the dataframe containing correction agent data.

    Parameters:
    df (pd.DataFrame): Input dataframe with correction agent results

    Returns:
    pd.DataFrame: DataFrame with extracted metrics
    """

    # Create a copy of the dataframe to avoid modifying the original
    metrics_df = df.copy()

    metrics_df['success_rate'] = True
    metrics_df['commit_count'] = metrics_df['commit_count']
    metrics_df['GLOC'] = metrics_df['global_line_changes']
    metrics_df['TLOC'] = metrics_df['total_line_changes']
    metrics_df['GHC'] = metrics_df['global_hunk_changes']
    metrics_df['GFC'] = metrics_df['global_file_changes']
    metrics_df['GAR-R'] = np.where(
        metrics_df['all_new_relevant_artifacts'] > 0,
        metrics_df['new_relevant_artifacts_updated'] / metrics_df['all_new_relevant_artifacts'],
        0
    )
    metrics_df['MAR-R'] = np.where(
        metrics_df['all_old_relevant_artifacts'] > 0,
        metrics_df['old_relevant_artifacts_updated'] / metrics_df['all_old_relevant_artifacts'],
        0
    )

    # Create formatted ratio columns
    # Generated Artifacts Ratio formatted as "x of y (z%)"
    metrics_df['GAR'] = metrics_df.apply(
        lambda row: format_rate_col(row, generated=True),
        axis=1
    )

    # Migrated Artifacts Ratio formatted as "x of y (z%)"
    metrics_df['MAR'] = metrics_df.apply(
        lambda row: format_rate_col(row, generated=False),
        axis=1
    )

    # Select the key metrics columns for the final result
    metrics_columns = [
        # Identifier columns
        'benchmark', 'app', 'run_name', 'microservice',

        # Core metrics
        'success_rate',
        'commit_count',
        'GLOC',
        'TLOC',
        'GHC',
        'GFC',
        'GAR',
        'MAR',

        # Formatted ratio columns
        # 'Generated_Artifacts_Ratio_Formatted',
        # 'Migrated_Artifacts_Ratio_Formatted',
        'new_relevant_artifacts_updated',
        'all_new_relevant_artifacts',
        'GAR-R',
        'old_relevant_artifacts_updated',
        'all_old_relevant_artifacts',
        'MAR-R',
    ]

    return metrics_df[metrics_columns]

In [119]:
metrics_df = extract_metrics(df)

In [120]:
metrics_df["key"] = metrics_df.apply(lambda row: f"{row['benchmark']}/{row['app']}/{row['run_name']}/{row['microservice']}", axis=1)
metrics_df = metrics_df.set_index("key")
metrics_df

,benchmark,app,run_name,microservice,success_rate,commit_count,GLOC,TLOC,GHC,GFC,GAR,MAR,new_relevant_artifacts_updated,all_new_relevant_artifacts,GAR-R,old_relevant_artifacts_updated,all_old_relevant_artifacts,MAR-R
key,,,,,,,,,,,,,,,,,,
monomorph/7ep-demo/single_run/authentication,monomorph,7ep-demo,single_run,authentication,True,5,66,68,0,5,3 of 21 (14.3%),1 of 41 (2.4%),3,21,0.142857,1,41,0.024390
monomorph/7ep-demo/single_run/library,monomorph,7ep-demo,single_run,library,True,3,85,91,2,6,4 of 17 (23.5%),2 of 42 (4.8%),4,17,0.235294,2,42,0.047619
monomorph/7ep-demo/single_run/mathematics,monomorph,7ep-demo,single_run,mathematics,True,0,0,0,0,0,0 of 0 (0.0%),0 of 26 (0.0%),0,0,0.000000,0,26,0.000000
monomorph/7ep-demo/single_run/persistence,monomorph,7ep-demo,single_run,persistence,True,6,39,39,0,5,2 of 13 (15.4%),3 of 27 (11.1%),2,13,0.153846,3,27,0.111111
monomorph/jpetstore-6/single_run/account,monomorph,jpetstore-6,single_run,account,True,19,6573,7093,86,91,19 of 27 (70.4%),20 of 20 (100.0%),19,27,0.703704,20,20,1.000000
monomorph/jpetstore-6/single_run/catalog,monomorph,jpetstore-6,single_run,catalog,True,13,5385,5593,95,98,23 of 36 (63.9%),23 of 23 (100.0%),23,36,0.638889,23,23,1.000000
monomorph/jpetstore-6/single_run/order,monomorph,jpetstore-6,single_run,order,True,46,9750,10754,101,107,23 of 31 (74.2%),34 of 34 (100.0%),23,31,0.741935,34,34,1.000000
monomorph/jpetstore-6/order_manual/order,monomorph,jpetstore-6,order_manual,order,True,8,324,324,0,15,9 of 31 (29.0%),6 of 34 (17.6%),9,31,0.290323,6,34,0.176471
monomorph/spring-petclinic/single_run/customers,monomorph,spring-petclinic,single_run,customers,True,6,6412,6452,16,37,22 of 34 (64.7%),9 of 28 (32.1%),22,34,0.647059,9,28,0.321429


In [121]:
for _, row in metrics_df[~metrics_df.run_name.isin(["single_run", "excluding_bloated_commits"])].iterrows():
    key =  f"{row['benchmark']}/{row['app']}/single_run/{row['microservice']}"
    metrics_df.loc[key, "success_rate"] = False
    key =  f"{row['benchmark']}/{row['app']}/excluding_bloated_commits/{row['microservice']}"
    metrics_df.loc[key, "success_rate"] = False
metrics_df

,benchmark,app,run_name,microservice,success_rate,commit_count,GLOC,TLOC,GHC,GFC,GAR,MAR,new_relevant_artifacts_updated,all_new_relevant_artifacts,GAR-R,old_relevant_artifacts_updated,all_old_relevant_artifacts,MAR-R
key,,,,,,,,,,,,,,,,,,
monomorph/7ep-demo/single_run/authentication,monomorph,7ep-demo,single_run,authentication,True,5,66,68,0,5,3 of 21 (14.3%),1 of 41 (2.4%),3,21,0.142857,1,41,0.024390
monomorph/7ep-demo/single_run/library,monomorph,7ep-demo,single_run,library,True,3,85,91,2,6,4 of 17 (23.5%),2 of 42 (4.8%),4,17,0.235294,2,42,0.047619
monomorph/7ep-demo/single_run/mathematics,monomorph,7ep-demo,single_run,mathematics,True,0,0,0,0,0,0 of 0 (0.0%),0 of 26 (0.0%),0,0,0.000000,0,26,0.000000
monomorph/7ep-demo/single_run/persistence,monomorph,7ep-demo,single_run,persistence,True,6,39,39,0,5,2 of 13 (15.4%),3 of 27 (11.1%),2,13,0.153846,3,27,0.111111
monomorph/jpetstore-6/single_run/account,monomorph,jpetstore-6,single_run,account,True,19,6573,7093,86,91,19 of 27 (70.4%),20 of 20 (100.0%),19,27,0.703704,20,20,1.000000
monomorph/jpetstore-6/single_run/catalog,monomorph,jpetstore-6,single_run,catalog,True,13,5385,5593,95,98,23 of 36 (63.9%),23 of 23 (100.0%),23,36,0.638889,23,23,1.000000
monomorph/jpetstore-6/single_run/order,monomorph,jpetstore-6,single_run,order,False,46,9750,10754,101,107,23 of 31 (74.2%),34 of 34 (100.0%),23,31,0.741935,34,34,1.000000
monomorph/jpetstore-6/order_manual/order,monomorph,jpetstore-6,order_manual,order,True,8,324,324,0,15,9 of 31 (29.0%),6 of 34 (17.6%),9,31,0.290323,6,34,0.176471
monomorph/spring-petclinic/single_run/customers,monomorph,spring-petclinic,single_run,customers,True,6,6412,6452,16,37,22 of 34 (64.7%),9 of 28 (32.1%),22,34,0.647059,9,28,0.321429


In [122]:
key = "monomorph/spring-petclinic/extended_run_vets/vets"
metrics_df.loc[key, "success_rate"] = False

In [123]:
save_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/refact-effort/monomorph_filtered_metrics.csv"
metrics_df.to_csv(save_path, index=False)

In [99]:
import pandas as pd

save_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/refact-effort/monomorph_filtered_metrics.csv"
metrics_df = pd.read_csv(save_path)

In [2]:
metrics_df

,benchmark,app,run_name,microservice,success_rate,commit_count,GLOC,TLOC,GHC,GFC,GAR,MAR
0,monomorph,7ep-demo,single_run,authentication,True,5,66,68,0,5,3 of 21 (14.3%),1 of 41 (2.4%)
1,monomorph,7ep-demo,single_run,library,True,3,85,91,2,6,4 of 17 (23.5%),2 of 42 (4.8%)
2,monomorph,7ep-demo,single_run,mathematics,True,0,0,0,0,0,0 of 0 (0.0%),0 of 26 (0.0%)
3,monomorph,7ep-demo,single_run,persistence,True,6,39,39,0,5,2 of 13 (15.4%),3 of 27 (11.1%)
4,monomorph,jpetstore-6,single_run,account,True,19,6573,7093,86,91,19 of 27 (70.4%),20 of 20 (100.0%)
5,monomorph,jpetstore-6,single_run,catalog,True,13,5385,5593,95,98,23 of 36 (63.9%),23 of 23 (100.0%)
6,monomorph,jpetstore-6,single_run,order,False,46,9750,10754,101,107,23 of 31 (74.2%),34 of 34 (100.0%)
7,monomorph,jpetstore-6,order_manual,order,True,7,170,170,0,13,9 of 31 (29.0%),4 of 34 (11.8%)
8,monomorph,spring-petclinic,single_run,customers,True,6,6412,6452,16,37,22 of 34 (64.7%),9 of 28 (32.1%)
9,monomorph,spring-petclinic,single_run,vets,False,6,5267,5401,17,31,20 of 29 (69.0%),9 of 28 (32.1%)


In [12]:
metrics_df.columns

Index(['benchmark', 'app', 'run_name', 'microservice', 'success_rate',
       'commit_count', 'GLOC', 'TLOC', 'GHC', 'GFC', 'GAR', 'MAR',
       'new_relevant_artifacts_updated', 'all_new_relevant_artifacts', 'GAR-R',
       'old_relevant_artifacts_updated', 'all_old_relevant_artifacts',
       'MAR-R'],
      dtype='object')

In [64]:
# single run metrics
single_run_metrics = metrics_df[metrics_df.run_name=="single_run"].reset_index().drop(columns=["key", "benchmark", "run_name", "GHC"])
num_cols = ['success_rate',
       'commit_count', 'GLOC', 'TLOC', 'GFC',
       'new_relevant_artifacts_updated', 'all_new_relevant_artifacts', 'GAR-R',
       'old_relevant_artifacts_updated', 'all_old_relevant_artifacts',
       'MAR-R']
single_run_metrics[num_cols]
mean_overall_line = single_run_metrics[num_cols].mean()
mean_overall_line

success_rate                         0.800000
commit_count                        12.200000
GLOC                              3876.100000
TLOC                              4113.900000
GFC                                 41.300000
new_relevant_artifacts_updated      13.800000
all_new_relevant_artifacts          23.200000
GAR-R                                0.486991
old_relevant_artifacts_updated      11.100000
all_old_relevant_artifacts          28.900000
MAR-R                                0.432598
dtype: float64

In [105]:
def get_overall_row(dfm):
    sum_cols = ['new_relevant_artifacts_updated', 'all_new_relevant_artifacts', 'old_relevant_artifacts_updated', 'all_old_relevant_artifacts']
    mean_cols = [
        'success_rate',  'GAR-R',  'MAR-R',
    ]
    median_cols = ['commit_count', 'GLOC', 'TLOC', 'GFC']
    overall_row = pd.Series(['Overall', "Overall"] + [None]*(len(dfm.columns)-2), index=dfm.columns)
    overall_row[sum_cols] = dfm[sum_cols].sum()
    overall_row[mean_cols] = dfm[mean_cols].mean()
    overall_row[median_cols] = dfm[median_cols].median()
    overall_row['GAR'] = format_rate_col(overall_row, generated=True)
    overall_row['MAR'] = format_rate_col(overall_row, generated=False)
    return overall_row

In [124]:
single_run_metrics = metrics_df[metrics_df.run_name=="single_run"].reset_index().drop(columns=["key", "benchmark", "run_name", "GHC"])
overall_row = get_overall_row(single_run_metrics)
single_run_metrics = pd.concat([single_run_metrics, overall_row.to_frame().T], axis=0)
single_run_metrics

,app,microservice,success_rate,commit_count,GLOC,TLOC,GFC,GAR,MAR,new_relevant_artifacts_updated,all_new_relevant_artifacts,GAR-R,old_relevant_artifacts_updated,all_old_relevant_artifacts,MAR-R
0,7ep-demo,authentication,True,5,66,68,5,3 of 21 (14.3%),1 of 41 (2.4%),3,21,0.142857,1,41,0.02439
1,7ep-demo,library,True,3,85,91,6,4 of 17 (23.5%),2 of 42 (4.8%),4,17,0.235294,2,42,0.047619
2,7ep-demo,mathematics,True,0,0,0,0,0 of 0 (0.0%),0 of 26 (0.0%),0,0,0.0,0,26,0.0
3,7ep-demo,persistence,True,6,39,39,5,2 of 13 (15.4%),3 of 27 (11.1%),2,13,0.153846,3,27,0.111111
4,jpetstore-6,account,True,19,6573,7093,91,19 of 27 (70.4%),20 of 20 (100.0%),19,27,0.703704,20,20,1.0
5,jpetstore-6,catalog,True,13,5385,5593,98,23 of 36 (63.9%),23 of 23 (100.0%),23,36,0.638889,23,23,1.0
6,jpetstore-6,order,False,46,9750,10754,107,23 of 31 (74.2%),34 of 34 (100.0%),23,31,0.741935,34,34,1.0
7,spring-petclinic,customers,True,6,6412,6452,37,22 of 34 (64.7%),9 of 28 (32.1%),22,34,0.647059,9,28,0.321429
8,spring-petclinic,vets,False,6,5267,5401,31,20 of 29 (69.0%),9 of 28 (32.1%),20,29,0.689655,9,28,0.321429
9,spring-petclinic,visits,True,18,5184,5648,33,22 of 24 (91.7%),10 of 20 (50.0%),22,24,0.916667,10,20,0.5


In [125]:
single_run_metrics = metrics_df[metrics_df.run_name=="single_run"].reset_index().drop(columns=["key", "benchmark", "run_name", "GHC"])
overall_row = get_overall_row(single_run_metrics)
single_run_metrics = pd.concat([single_run_metrics, overall_row.to_frame().T], axis=0)
single_run_metrics
# format the table for latex
single_run_metrics[["commit_count", "TLOC", "GFC"]] = single_run_metrics[["commit_count", "TLOC", "GFC"]].astype(int)
# single_run_metrics.drop(columns=['new_relevant_artifacts_updated', 'all_new_relevant_artifacts', 'GAR-R',
#        'old_relevant_artifacts_updated', 'all_old_relevant_artifacts',
#        'MAR-R'])
single_run_metrics = single_run_metrics.rename(columns={
    'success_rate': 'Success Rate',
    'commit_count': 'Commit Count',
    'GLOC': 'GLOC',
    'TLOC': 'TLOC',
    'GFC': 'GFC',
    'GAR': 'GAR',
    'MAR': 'MAR'
})
single_run_metrics = single_run_metrics[['app', 'microservice', 'Success Rate', 'Commit Count', 'GLOC', 'TLOC', 'GFC', 'GAR', 'MAR']]
single_run_metrics = single_run_metrics.reset_index(drop=True)
single_run_metrics.replace(
    {
        'app': {
            'spring-petclinic': 'PetClinic',
            'jpetstore-6': 'JPetStore',
            '7ep-demo': '7ep Demo',
        }
    },
    inplace=True
)
single_run_metrics["microservice"] = single_run_metrics["microservice"].apply(lambda x: x.title())
single_run_metrics = single_run_metrics.rename(columns={'microservice': 'Microservice', 'app': 'Application'})
single_run_metrics = single_run_metrics.set_index(['Application', 'Microservice'])
single_run_metrics["Success Rate"] = single_run_metrics["Success Rate"].apply(lambda x: f"{int(x*100)}\\%")
single_run_metrics = single_run_metrics.style.format({
    # 'Success Rate': '{:.1%}',
    'Commit Count': '{:,.0f}',
    'GLOC': '{:,.0f}',
    'TLOC': '{:,.0f}',
    'GFC': '{:,.0f}',
    'GAR': lambda x: x.replace('%', '\\%'),
    'MAR': lambda x: x.replace('%', '\\%'),
})
single_run_metrics

In [85]:
single_run_metrics_latex_str = single_run_metrics.to_latex(
    hrules=True,
    label="tab-mm:single_run_metrics",
    caption="MonoMorph Correction Agent Initial Run Effort Metrics",
    position="htbp",
    column_format="ll|ccccccc",
)
print(single_run_metrics_latex_str)

\begin{table}[htbp]
\caption{MonoMorph Correction Agent Initial Run Effort Metrics}
\label{tab-mm:single_run_metrics}
\begin{tabular}{lcccccc}
\toprule
{} & {} & {Success Rate} & {Commit Count} & {GLOC} & {TLOC} & {GFC} & {GAR} & {MAR} \\
{Application} & {Microservice} & {} & {} & {} & {} & {} & {} & {} \\
\midrule
\multirow[c]{4}{*}{7ep Demo} & Authentication & 100\% & 5 & 66 & 68 & 5 & 3 of 21 (14.3\%) & 1 of 41 (2.4\%) \\
 & Library & 100\% & 3 & 85 & 91 & 6 & 4 of 17 (23.5\%) & 2 of 42 (4.8\%) \\
 & Mathematics & 100\% & 0 & 0 & 0 & 0 & 0 of 0 (0.0\%) & 0 of 26 (0.0\%) \\
 & Persistence & 100\% & 6 & 39 & 39 & 5 & 2 of 13 (15.4\%) & 3 of 27 (11.1\%) \\
\multirow[c]{3}{*}{JPetStore} & Account & 100\% & 19 & 6,573 & 7,093 & 91 & 19 of 27 (70.4\%) & 20 of 20 (100.0\%) \\
 & Catalog & 100\% & 13 & 5,385 & 5,593 & 98 & 23 of 36 (63.9\%) & 23 of 23 (100.0\%) \\
 & Order & 0\% & 46 & 9,750 & 10,754 & 107 & 23 of 31 (74.2\%) & 34 of 34 (100.0\%) \\
\multirow[c]{3}{*}{PetClinic} & Customers

In [86]:
pd.DataFrame.to_latex?

In [37]:
single_run_metrics2.loc[without_bloat_rows.index]

,app,microservice,success_rate,commit_count,GLOC,TLOC,GFC,GAR,MAR,new_relevant_artifacts_updated,all_new_relevant_artifacts,GAR-R,old_relevant_artifacts_updated,all_old_relevant_artifacts,MAR-R
key,,,,,,,,,,,,,,,
monomorph/jpetstore-6/account,jpetstore-6,account,True,19,6573,7093,91,19 of 27 (70.4%),20 of 20 (100.0%),19,27,0.703704,20,20,1.000000
monomorph/jpetstore-6/catalog,jpetstore-6,catalog,True,13,5385,5593,98,23 of 36 (63.9%),23 of 23 (100.0%),23,36,0.638889,23,23,1.000000
monomorph/jpetstore-6/order,jpetstore-6,order,False,46,9750,10754,107,23 of 31 (74.2%),34 of 34 (100.0%),23,31,0.741935,34,34,1.000000
monomorph/spring-petclinic/customers,spring-petclinic,customers,True,6,6412,6452,37,22 of 34 (64.7%),9 of 28 (32.1%),22,34,0.647059,9,28,0.321429
monomorph/spring-petclinic/vets,spring-petclinic,vets,False,6,5267,5401,31,20 of 29 (69.0%),9 of 28 (32.1%),20,29,0.689655,9,28,0.321429
monomorph/spring-petclinic/visits,spring-petclinic,visits,True,18,5184,5648,33,22 of 24 (91.7%),10 of 20 (50.0%),22,24,0.916667,10,20,0.500000


In [126]:
single_run_metrics2 = metrics_df[metrics_df.run_name=="single_run"].drop(columns=["benchmark", "run_name", "GHC"])
new_index = single_run_metrics2.index.str.replace("single_run/", "")
single_run_metrics2.index = new_index
without_bloat_rows = metrics_df[metrics_df.run_name=="excluding_bloated_commits"].drop(columns=["benchmark", "run_name", "GHC"])
without_bloat_rows.index = without_bloat_rows.index.str.replace("excluding_bloated_commits/", "")
discarded_commits = (single_run_metrics2["commit_count"] - without_bloat_rows["commit_count"]).fillna(0)
single_run_metrics2.loc[without_bloat_rows.index] = without_bloat_rows
without_bloat_metrics = single_run_metrics2
without_bloat_metrics["discarded_commits"] = discarded_commits
without_bloat_metrics = without_bloat_metrics.reset_index(drop=True)
overall_row = get_overall_row(without_bloat_metrics)
overall_row['discarded_commits'] = without_bloat_metrics['discarded_commits'].sum()
without_bloat_metrics = pd.concat([without_bloat_metrics, overall_row.to_frame().T], axis=0)
without_bloat_metrics

,app,microservice,success_rate,commit_count,GLOC,TLOC,GFC,GAR,MAR,new_relevant_artifacts_updated,all_new_relevant_artifacts,GAR-R,old_relevant_artifacts_updated,all_old_relevant_artifacts,MAR-R,discarded_commits
0,7ep-demo,authentication,True,5,66,68,5,3 of 21 (14.3%),1 of 41 (2.4%),3,21,0.142857,1,41,0.02439,0.0
1,7ep-demo,library,True,3,85,91,6,4 of 17 (23.5%),2 of 42 (4.8%),4,17,0.235294,2,42,0.047619,0.0
2,7ep-demo,mathematics,True,0,0,0,0,0 of 0 (0.0%),0 of 26 (0.0%),0,0,0.0,0,26,0.0,0.0
3,7ep-demo,persistence,True,6,39,39,5,2 of 13 (15.4%),3 of 27 (11.1%),2,13,0.153846,3,27,0.111111,0.0
4,jpetstore-6,account,True,17,6573,797,91,6 of 27 (22.2%),8 of 20 (40.0%),6,27,0.222222,8,20,0.4,2.0
5,jpetstore-6,catalog,True,12,5385,251,98,7 of 36 (19.4%),4 of 23 (17.4%),7,36,0.194444,4,23,0.173913,1.0
6,jpetstore-6,order,False,44,9750,1350,107,11 of 31 (35.5%),11 of 34 (32.4%),11,31,0.354839,11,34,0.323529,2.0
7,spring-petclinic,customers,True,5,6412,1142,37,7 of 34 (20.6%),5 of 28 (17.9%),7,34,0.205882,5,28,0.178571,1.0
8,spring-petclinic,vets,False,5,5267,433,31,5 of 29 (17.2%),4 of 28 (14.3%),5,29,0.172414,4,28,0.142857,1.0
9,spring-petclinic,visits,True,17,5184,1633,33,14 of 24 (58.3%),9 of 20 (45.0%),14,24,0.583333,9,20,0.45,1.0


In [127]:
# format the table for latex
without_bloat_metrics[["TLOC", "GFC"]] = without_bloat_metrics[["TLOC", "GFC"]].astype(int)
# without_bloat_metrics.drop(columns=['new_relevant_artifacts_updated', 'all_new_relevant_artifacts', 'GAR-R',
#        'old_relevant_artifacts_updated', 'all_old_relevant_artifacts',
#        'MAR-R'])
without_bloat_metrics = without_bloat_metrics.rename(columns={
    'success_rate': 'Success Rate',
    'commit_count': 'Commit Count',
    'GLOC': 'GLOC',
    'TLOC': 'TLOC',
    'GFC': 'GFC',
    'GAR': 'GAR',
    'MAR': 'MAR',
    'discarded_commits': 'Discarded Commits'
})
without_bloat_metrics = without_bloat_metrics[['app', 'microservice', 'Success Rate', 'Commit Count', 'GLOC', 'TLOC', 'GFC', 'GAR', 'MAR', 'Discarded Commits']]
without_bloat_metrics = without_bloat_metrics.reset_index(drop=True)
without_bloat_metrics.replace(
    {
        'app': {
            'spring-petclinic': 'PetClinic',
            'jpetstore-6': 'JPetStore',
            '7ep-demo': '7ep Demo',
        }
    },
    inplace=True
)
without_bloat_metrics["microservice"] = without_bloat_metrics["microservice"].apply(lambda x: x.title())
without_bloat_metrics["Success Rate"] = without_bloat_metrics["Success Rate"].apply(lambda x: f"{int(x*100)}\\%")
without_bloat_metrics = without_bloat_metrics.rename(columns={'microservice': 'Microservice', 'app': 'Application'})
without_bloat_metrics = without_bloat_metrics.set_index(['Application', 'Microservice'])
without_bloat_metrics = without_bloat_metrics.style.format({
    # 'Success Rate': '{:.1%}',
    'Commit Count': '{:,.0f}',
    'GLOC': '{:,.0f}',
    'TLOC': '{:,.0f}',
    'GFC': '{:,.0f}',
    'GAR': lambda x: x.replace('%', '\\%'),
    'MAR': lambda x: x.replace('%', '\\%'),
    'Discarded Commits': '{:,.0f}'
})
without_bloat_metrics

In [89]:
without_bloat_metrics_latex_str = without_bloat_metrics.to_latex(
    hrules=True,
    label="tab-mm:without_bloat_metrics",
    caption="MonoMorph Correction Agent Initial Run Effort Metrics (Excluding 'Cosmetic' Commits)",
    position="htbp",
    column_format="ll|cccccccc",
)
print(without_bloat_metrics_latex_str)

\begin{table}[htbp]
\caption{MonoMorph Correction Agent Initial Run Effort Metrics (Excluding 'Cosmetic' Commits)}
\label{tab-mm:without_bloat_metrics}
\begin{tabular}{ll|cccccccc}
\toprule
{} & {} & {Success Rate} & {Commit Count} & {GLOC} & {TLOC} & {GFC} & {GAR} & {MAR} & {Discarded Commits} \\
{Application} & {Microservice} & {} & {} & {} & {} & {} & {} & {} & {} \\
\midrule
\multirow[c]{4}{*}{7ep Demo} & Authentication & 100\% & 5 & 66 & 68 & 5 & 3 of 21 (14.3\%) & 1 of 41 (2.4\%) & 0 \\
 & Library & 100\% & 3 & 85 & 91 & 6 & 4 of 17 (23.5\%) & 2 of 42 (4.8\%) & 0 \\
 & Mathematics & 100\% & 0 & 0 & 0 & 0 & 0 of 0 (0.0\%) & 0 of 26 (0.0\%) & 0 \\
 & Persistence & 100\% & 6 & 39 & 39 & 5 & 2 of 13 (15.4\%) & 3 of 27 (11.1\%) & 0 \\
\multirow[c]{3}{*}{JPetStore} & Account & 100\% & 17 & 6,573 & 797 & 91 & 6 of 27 (22.2\%) & 8 of 20 (40.0\%) & 2 \\
 & Catalog & 100\% & 12 & 5,385 & 251 & 98 & 7 of 36 (19.4\%) & 4 of 23 (17.4\%) & 1 \\
 & Order & 0\% & 44 & 9,750 & 1,350 & 107 & 11 of

In [90]:
metrics_df.run_name.unique()

array(['single_run', 'order_manual', 'extended_run_vets',
       'excluding_bloated_commits'], dtype=object)

In [91]:
# Results for the manual run and the extended run

additional_metrics = metrics_df[metrics_df.run_name.isin(["order_manual", "extended_run_vets"])].reset_index(drop=True).drop(columns=["benchmark", "run_name", "GHC", 'success_rate'])
additional_metrics

,app,microservice,success_rate,commit_count,GLOC,TLOC,GFC,GAR,MAR,new_relevant_artifacts_updated,all_new_relevant_artifacts,GAR-R,old_relevant_artifacts_updated,all_old_relevant_artifacts,MAR-R
0,jpetstore-6,order,True,5,137,137,11,8 of 31 (25.8%),3 of 34 (8.8%),8,31,0.258065,3,34,0.088235
1,spring-petclinic,vets,False,1,21,21,2,2 of 29 (6.9%),0 of 28 (0.0%),2,29,0.068966,0,28,0.000000


In [128]:
# Results for the manual run and the extended run

additional_metrics = metrics_df[metrics_df.run_name.isin(["order_manual", "extended_run_vets"])].reset_index(drop=True).drop(columns=["benchmark", "run_name", "GHC", 'success_rate'])
additional_metrics

,app,microservice,commit_count,GLOC,TLOC,GFC,GAR,MAR,new_relevant_artifacts_updated,all_new_relevant_artifacts,GAR-R,old_relevant_artifacts_updated,all_old_relevant_artifacts,MAR-R
0,jpetstore-6,order,8,324,324,15,9 of 31 (29.0%),6 of 34 (17.6%),9,31,0.290323,6,34,0.176471
1,spring-petclinic,vets,1,21,21,2,2 of 29 (6.9%),0 of 28 (0.0%),2,29,0.068966,0,28,0.000000


In [129]:
# format the table for latex
additional_metrics[["TLOC", "GFC"]] = additional_metrics[["TLOC", "GFC"]].astype(int)
# without_bloat_metrics.drop(columns=['new_relevant_artifacts_updated', 'all_new_relevant_artifacts', 'GAR-R',
#        'old_relevant_artifacts_updated', 'all_old_relevant_artifacts',
#        'MAR-R'])
additional_metrics = additional_metrics.rename(columns={
    # 'success_rate': 'Success Rate',
    'commit_count': 'Commit Count',
    'GLOC': 'GLOC',
    'TLOC': 'TLOC',
    'GFC': 'GFC',
    'GAR': 'GAR',
    'MAR': 'MAR'
})
additional_metrics = additional_metrics[['app', 'microservice', 'Commit Count', 'GLOC', 'TLOC', 'GFC', 'GAR', 'MAR']]
additional_metrics = additional_metrics.reset_index(drop=True)
additional_metrics.replace(
    {
        'app': {
            'spring-petclinic': 'PetClinic',
            'jpetstore-6': 'JPetStore',
            '7ep-demo': '7ep Demo',
        }
    },
    inplace=True
)
additional_metrics["microservice"] = additional_metrics["microservice"].apply(lambda x: x.title())
# additional_metrics["Success Rate"] = additional_metrics["Success Rate"].apply(lambda x: f"{int(x*100)}\\%")
additional_metrics = additional_metrics.rename(columns={'microservice': 'Microservice', 'app': 'Application'})
additional_metrics = additional_metrics.set_index(['Application', 'Microservice'])
additional_metrics = additional_metrics.style.format({
    # 'Success Rate': '{:.1%}',
    'Commit Count': '{:,.0f}',
    'GLOC': '{:,.0f}',
    'TLOC': '{:,.0f}',
    'GFC': '{:,.0f}',
    'GAR': lambda x: x.replace('%', '\\%'),
    'MAR': lambda x: x.replace('%', '\\%'),
    # 'Discarded Commits': '{:,.0f}'
})
additional_metrics

,,Commit Count,GLOC,TLOC,GFC,GAR,MAR
Application,Microservice,,,,,,
JPetStore,Order,8,324,324,15,9 of 31 (29.0\%),6 of 34 (17.6\%)
PetClinic,Vets,1,21,21,2,2 of 29 (6.9\%),0 of 28 (0.0\%)


In [130]:
additional_metrics_latex_str = additional_metrics.to_latex(
    hrules=True,
    label="tab-mm:additional_effort_metrics",
    caption="Additional Run (PetClinic Vets) and Manual Correction (JPetStore Order) Effort Metrics",
    position="htbp",
    column_format="ll|cccccc",
)
print(additional_metrics_latex_str)

\begin{table}[htbp]
\caption{Additional Run (PetClinic Vets) and Manual Correction (JPetStore Order) Effort Metrics}
\label{tab-mm:additional_effort_metrics}
\begin{tabular}{ll|cccccc}
\toprule
{} & {} & {Commit Count} & {GLOC} & {TLOC} & {GFC} & {GAR} & {MAR} \\
{Application} & {Microservice} & {} & {} & {} & {} & {} & {} \\
\midrule
JPetStore & Order & 8 & 324 & 324 & 15 & 9 of 31 (29.0\%) & 6 of 34 (17.6\%) \\
PetClinic & Vets & 1 & 21 & 21 & 2 & 2 of 29 (6.9\%) & 0 of 28 (0.0\%) \\
\bottomrule
\end{tabular}
\end{table}

